# Cognopolis · Урок M1 — реактивный житель-сборщик (домашка)

Домашка к [Уроку M1](https://itrubnikov.github.io/Train_of_Thought/docs/game-lessons/m1-reactive-gatherer/):
собери **сам** такого же простейшего агента, какого мы собирали в воркспейсе
(`workspace.ipynb` в этой папке). Житель ходит по карте, добывает дерево и камень, а с
полным рюкзаком возвращается домой — дом сам принимает добычу на склад.

Всё общение с миром — **голый REST API** (никаких установок клиента игры): агент видит мир
только через HTTP-ответы. Ноутбук working-first: `Run all` проходит целиком прямо сейчас,
базовая версия агента уже работает — твои задачи делают её по-настоящему **реактивной**.


## 1. Сетап

In [ ]:
%pip install -q requests

In [ ]:
import os

def read_secret(name: str, default: str = "") -> str:
    """Секрет из Colab userdata -> Kaggle Secrets -> переменной окружения -> default."""
    try:
        from google.colab import userdata          # Colab: Secrets на панели слева
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle: Add-ons -> Secrets
        v = UserSecretsClient().get_secret(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, default)

# ЖИВОЙ МИР COGNOPOLIS (публичный инстанс). Свой сервер: секрет/env COGNOPOLIS_URL.
BASE_URL = read_secret("COGNOPOLIS_URL", "https://kindomklaster.com").rstrip("/")

# ТВОЙ ТОКЕН (полный доступ к твоему жителю):
#   1. Открой BASE_URL в браузере и зарегистрируйся (логин + пароль).
#   2. Ратуша -> вкладка «аккаунт» -> «копировать» — это твой токен.
#   3. Положи его в секрет COGNOPOLIS_TOKEN (Colab/Kaggle) или в переменную окружения.
TOKEN = read_secret("COGNOPOLIS_TOKEN")
assert TOKEN, f"Вставь токен: зарегистрируйся на {BASE_URL}, скопируй токен из Ратуши и задай COGNOPOLIS_TOKEN."
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

print("Мир:", BASE_URL)
print("Смотри за жителем в браузере:", f"{BASE_URL}/?token={TOKEN[:6]}...  (полная ссылка = BASE_URL/?token=<твой токен>)")


## 2. Разогрев — мини-клиент из 30 строк

Весь «SDK» урока — класс `Game` ниже: три GET («глаза») и два POST («руки») поверх
`requests`. Правила игры целиком на стороне сервера: нарушение приходит ошибкой
`{"error": {"code", "message"}}` — мы поднимаем её как `GameError` с честным кодом
(`no_resource_here`, `inventory_full`, `character_on_cooldown`...). Полный API — на
`BASE_URL/docs` (кнопка Authorize, токен без префикса Bearer).


In [ ]:
import time

import requests


class GameError(Exception):
    """Отказ сервера по правилам игры; .code — стабильный код причины."""

    def __init__(self, code: str, message: str):
        super().__init__(f"{code}: {message}")
        self.code = code


class Game:
    """Мини-клиент Cognopolis поверх REST: get_character/get_map/get_assignment + move_dir/gather."""

    def __init__(self, base_url: str, token: str):
        self.base = base_url
        self.headers = {"Authorization": f"Bearer {token}"}
        self.last_cooldown = 0.0

    def _call(self, method: str, path: str, **body) -> dict:
        r = requests.request(method, self.base + path, headers=self.headers,
                             json=body or None, timeout=15)
        data = r.json()
        if not r.ok:
            err = data.get("error") or {}
            raise GameError(err.get("code", str(r.status_code)), err.get("message", r.text))
        if "cooldown" in data:                      # действия возвращают кулдаун — запоминаем
            self.last_cooldown = float(data["cooldown"])
        return data

    # --- глаза (observe) ---
    def get_character(self) -> dict:
        return self._call("GET", "/character")

    def get_map(self) -> dict:
        return self._call("GET", "/map")

    def get_assignment(self) -> dict:
        return self._call("GET", "/assignment")

    # --- руки (act) ---
    def move_dir(self, direction: str, reason: str | None = None) -> dict:
        """Шаг в направлении: north/south/east/west + 4 диагонали (D-069)."""
        return self._call("POST", f"/actions/move/{direction}", reason=reason)

    def gather(self, reason: str | None = None) -> dict:
        """Добыча на СВОЕЙ клетке (встань на tree/rock, потом добывай)."""
        return self._call("POST", "/actions/gather", reason=reason)

    # --- ритм (wait) ---
    def wait_cooldown(self) -> None:
        time.sleep(self.last_cooldown)
        self.last_cooldown = 0.0


g = Game(BASE_URL, TOKEN)

ch = g.get_character()
print("позиция:", (ch["x"], ch["y"]), "| рюкзак:", ch["inventory"], "| cap:", ch["inventory_cap"])

world = g.get_map()
print("карта", world["size"], "×", world["size"], "| на клетках:", sorted({t["content"] for t in world["tiles"]}))


Каждое действие возвращает результат **и кулдаун** — сколько секунд житель «занят».
Это естественный ритм петли: `observe -> decide -> act -> wait`. Спамить бессмысленно —
ранний повтор отбивается ошибкой `character_on_cooldown`, сервер — источник истины.


In [ ]:
# Одно действие: шаг в сторону (не в край карты). В ответе — новое состояние и cooldown.
time.sleep(ch["cooldown"])                                # житель мог быть «занят» после прошлых прогонов
direction = "east" if ch["x"] < world["size"] - 1 else "west"
res = g.move_dir(direction, reason="разогрев — пробую сходить")
print("cooldown:", res["cooldown"], "c | новая позиция:", (res["character"]["x"], res["character"]["y"]))
g.wait_cooldown()  # observe -> decide -> act -> ВОТ ЭТО ОЖИДАНИЕ

print("поручение от игрока:", g.get_assignment()["assignment"])   # None = житель свободен


## 3. Разбор — паттерн реактивного агента

Реактивный агент каждый ход решает **заново из текущего состояния** — маршрут не зашит:

- `observe` — `get_character()` + `get_map()` (+ `get_assignment()` — поручение от игрока);
- `decide` — правила на состояние: рюкзак полон -> домой `(0,0)` (дом сам разгрузит,
  отдельного действия «сдать» нет — в ответе `move` появится поле `banked`); иначе ->
  к ноде ресурса, которого меньше;
- `act` — ровно одно действие: шаг `move_dir` (8 направлений, диагонали экономят ходы)
  или `gather` **стоя на клетке ресурса** (D-069: добыча со своей клетки, не с соседней);
- `wait` — `wait_cooldown()`.

Вся «умность» — в `decide`; из пары условий рождается осмысленное поведение.


## 4. Задачи — собери своего реактивного сборщика

Каркас ниже уже работает (`Run all` проходит), но агент «туповат»: всегда идёт к
ближайшему дереву. Доведи `decide()` до реактивного — три задачи `# TODO` (каждая
со снятым комментарием остаётся рабочей):

1. **Разгрузка.** Рюкзак полон -> домой `(0,0)`; в логе появится `дом: авто-разгрузка`.
2. **Баланс.** Бери ресурс, которого меньше — есть готовый helper `scarcer_resource`.
3. **Поручение.** Выпиши жителю поручение в Ратуше (вкладка «поручения») и научи
   `decide()` его читать: если поручена добыча — добывай порученный ресурс.

**Задача со звёздочкой (после воркспейса).** Собери этого же агента на smolagents, как в
слое 4 воркспейса: оберни `move_dir`/`gather`/статус в `@tool` и дай LLM задачу «собери
2 дерева и вернись домой». Понадобится локальная модель (LM Studio/Ollama) или ключ MiniMax.


In [ ]:
RESOURCE_NODE = {"wood": "tree", "stone": "rock"}   # ресурс -> клетка, которая его даёт
HOME = (0, 0)   # дом: шаг на эту клетку авто-разгружает рюкзак на склад

STEP_DIR = {(1, 0): "east", (-1, 0): "west", (0, 1): "south", (0, -1): "north",
            (1, 1): "southeast", (1, -1): "northeast", (-1, 1): "southwest", (-1, -1): "northwest"}


def nearest(ch, tiles, content):
    """Ближайшая клетка с заданным содержимым (по манхэттенскому расстоянию)."""
    here = (ch["x"], ch["y"])
    nodes = [t for t in tiles if t["content"] == content]
    return min(nodes, key=lambda t: abs(t["x"] - here[0]) + abs(t["y"] - here[1]))


def on_tile(ch, tx, ty):
    """Стоим ли мы на клетке цели (добыча работает только со своей клетки, D-069)."""
    return (ch["x"], ch["y"]) == (tx, ty)


def carried_total(ch):
    return sum(ch["inventory"].values())


def scarcer_resource(ch):
    """Ресурс, которого у жителя меньше (рюкзак + склад) — кандидат в цель."""
    owned = {r: ch["inventory"].get(r, 0) + ch["stored"].get(r, 0) for r in RESOURCE_NODE}
    return min(RESOURCE_NODE, key=lambda r: owned[r])


In [ ]:
def decide(ch, world):
    """Верни кортеж (target_x, target_y, action, reason).
    action — что делать: "gather" (добыть, когда ВСТАНЕМ на клетку) или "home" (идти домой —
    дом сам разгрузит рюкзак, отдельного действия для этого нет).

    БАЗОВАЯ версия (работает, но не реактивная): всегда идём к ближайшему дереву и рубим.
    Доработай по «Задачам» выше.
    """
    # TODO 1 — разгрузка: если carried_total(ch) >= ch["inventory_cap"] — иди домой:
    #   return HOME[0], HOME[1], "home", "рюкзак полон — иду домой разгружаться"

    # TODO 2 — баланс: вместо жёсткого "wood" возьми want = scarcer_resource(ch)

    # TODO 3 — поручение: если игрок поручил добычу — исполняй её ресурс:
    #   a = g.get_assignment()["assignment"]
    #   if a and a["type"] == "gather" and a.get("resource"):
    #       want = a["resource"]  # (reason подскажет зрителям: f"по поручению: {want}")
    want = "wood"
    node = nearest(ch, world["tiles"], RESOURCE_NODE[want])
    return node["x"], node["y"], "gather", f"добываю {want} (базовая версия — улучшь меня!)"


def step_toward(ch, tx, ty):
    """Направление одного шага к цели (диагонали разрешены — так быстрее)."""
    dx = (tx > ch["x"]) - (tx < ch["x"])
    dy = (ty > ch["y"]) - (ty < ch["y"])
    return STEP_DIR[(dx, dy)]


def run(rounds=40):
    for _ in range(rounds):
        ch = g.get_character()                        # observe
        tx, ty, action, reason = decide(ch, world)    # decide
        if on_tile(ch, tx, ty):
            if action == "gather":
                try:                                  # act: стоим на ноде — добываем
                    g.gather(reason=reason)
                except GameError as e:
                    print("  gather blocked:", e.code)    # напр. inventory_full -> пора домой
                    time.sleep(1)                     # отказ не двигает кулдаун — не молотим сервер
            # action == "home" и мы дома: рюкзак уже разгрузился на заходе
        else:
            res = g.move_dir(step_toward(ch, tx, ty), reason=reason)   # act: шаг к цели
            banked = res["result"].get("banked")
            if banked:                                # ступили на дом — рюкзак сам ушёл на склад
                print("  дом: авто-разгрузка", banked)
        g.wait_cooldown()                             # wait


run()


## 5. Проверка

In [ ]:
ch = g.get_character()
total = sum(ch["inventory"].values()) + sum(ch["stored"].values())
skills = {k: v["level"] for k, v in ch["skills"].items()}
print("навыки:", skills, "| рюкзак:", ch["inventory"], "| склад:", ch["stored"], "| всего собрано:", total)

assert total >= 5, "Цель: собрать >= 5 ресурсов. Проверь decide() и число rounds."
print("Базовая цель достигнута — теперь сделай агента реактивным (домой-разгрузка + баланс + поручение).")


## Наблюдаемость — смотри за умом своего агента

Открой `BASE_URL/?token=<твой токен>` во второй вкладке: шаги жителя, мысль-пузырь
(`reason` каждого действия) и Хроника событий — в одной вкладке крутится цикл, в другой
живёт житель. «1 житель = 1 агент».

Когда база готова — вернись к **задаче со звёздочкой** (агент на smolagents): образец —
слой 4 воркспейса `workspace.ipynb`. LLM-«мозг» всерьёз разбираем в уроке M4.
